# Experiment 08: Deep Learning Application for Breast Cancer Classification

**Aim:** Implement a deep learning application for breast cancer classification using the Breast Cancer Wisconsin Diagnostic Dataset.

Educational use only; this model is not clinically validated and must not be used for medical decisions.

## 1. Import Libraries

Install dependencies with `pip install numpy pandas tensorflow matplotlib scikit-learn jupyter` if needed.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully")
print("TensorFlow version:", tf.__version__)

## 2. Load Dataset

Place `data.csv` in the same directory as this notebook. This relative path replaces the machine-specific path in the supplied document.

In [ ]:
DATA_PATH = "data.csv"
df = pd.read_csv(DATA_PATH)
print("Dataset loaded successfully")
print("Dataset shape:", df.shape)
display(df.head())

## 3. Prepare Input Features and Target

The target encoding follows the source experiment: malignant (`M`) = `1`, benign (`B`) = `0`. Identifier and optional unused columns are excluded.

In [ ]:
X = df.drop(columns=["id", "diagnosis", "Unnamed: 32"], errors="ignore")
y = df["diagnosis"].map({"M": 1, "B": 0})

if y.isna().any():
    raise ValueError("Unexpected or missing diagnosis labels were found.")

X = X.apply(pd.to_numeric, errors="coerce")
if X.isna().any().any():
    missing = X.isna().sum()
    raise ValueError("Missing/non-numeric feature values found:\n" + str(missing[missing > 0]))

print("Input shape:", X.shape)
print("Output shape:", y.shape)
print("Data prepared successfully")
print("Class counts (0=Benign, 1=Malignant):")
print(y.value_counts().sort_index())

## 4. Split Dataset

Use an 80/20 stratified split with `random_state=42`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

## 5. Standardize Features

Fit the scaler only on training data, then transform both sets.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Data normalization completed successfully")

## 6. Create and Compile the ANN

The network uses dense layers with ReLU activation, dropout regularization, and a sigmoid output.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## 7. Train the Model

Train for 30 epochs, batch size 16, and use 20% of the training data for validation.

In [ ]:
history = model.fit(
    X_train_scaled, y_train,
    epochs=30, batch_size=16,
    validation_split=0.2, verbose=1
)
print("Model training completed successfully")

## 8. Evaluate the Model

In [ ]:
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print("Test loss:", loss)
print("Test accuracy:", accuracy)
print("Test accuracy (%):", accuracy * 100)

## 9. Generate Predictions and Compare Results

A threshold of `0.5` converts sigmoid scores into binary predictions.

In [ ]:
y_pred_probability = model.predict(X_test_scaled, verbose=0).ravel()
y_pred = (y_pred_probability >= 0.5).astype(int)
comparison = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": y_pred
})
comparison["Actual label"] = comparison["Actual"].map({0: "Benign", 1: "Malignant"})
comparison["Predicted label"] = comparison["Predicted"].map({0: "Benign", 1: "Malignant"})
display(comparison.head(10))

## 10. Plot Training and Validation Accuracy

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Deep Learning Model Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 11. Reusable Prediction Function

Pass feature values in the same order as `X.columns`. The returned score is not a clinically validated probability.

In [ ]:
def predict_breast_cancer(features):
    features = np.asarray(features, dtype=float).reshape(1, -1)
    if features.shape[1] != X.shape[1]:
        raise ValueError(f"Expected {X.shape[1]} feature values, got {features.shape[1]}.")
    features_scaled = scaler.transform(features)
    probability_malignant = float(model.predict(features_scaled, verbose=0)[0][0])
    if probability_malignant >= 0.5:
        return "Malignant", probability_malignant * 100
    return "Benign", (1 - probability_malignant) * 100

print("Breast Cancer Classification Application Ready")

## 12. Test with a Dataset Record

In [ ]:
sample_features = X.iloc[0].to_numpy()
actual_diagnosis = df["diagnosis"].iloc[0]
prediction, confidence = predict_breast_cancer(sample_features)
print("Breast Cancer Classification Result")
print("------------------------------------")
print("Actual diagnosis:", actual_diagnosis)
print("Predicted diagnosis:", prediction)
print("Model score: {:.2f}%".format(confidence))

## Notes and Limitations

The supplied experiment document reports test loss of approximately `0.0789` and test accuracy of approximately `98.25%` for its recorded run. Results may vary when rerunning training. This is an academic demonstration only, not a clinical tool.